In [10]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [11]:
sqlite_index.count()

83

In [12]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I missed the first homework - can I still get a certificate?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
 'Homework: Why does the content keep changing?',
 'What happens to code saved in Codespaces if I do not commit it?']

In [13]:
from rag_helper import RAGBase
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

openai_client = OpenAI()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [14]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join after it started. If you want a certificate, you need to submit your project while submissions are still open.


In [18]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—usually you can, **if the course is still open and has space**.\n\nTo check, look for:\n- **Enrollment status**: open / closed / waitlist\n- **Start date**: whether it has already begun\n- **Registration deadline**: if there is one\n- **Prerequisites**: any required background\n- **Contact info**: the instructor or course admin\n\nIf you want, I can help you draft a short message to ask whether late enrollment is possible.'

In [19]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [20]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [25]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment late join"}', call_id='call_F836zV7NnXVD83wx5ACPZCT1', name='search', type='function_call', id='fc_0ab91f538aff5a81006a33cb32521c819998cf3600e15ace35', namespace=None, status='completed')]

In [26]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

NameError: name 'index' is not defined